### Restricted Kohn–Sham (RKS) tutorial

This tutorial shows how to use custom restricted Kohn–Sham algorithm `RKS.py`. 

#### Necessary imports

In [1]:
import sys
import psi4

%load_ext autoreload
%autoreload 2
sys.path.append('/Users/ivanbosko/Documents/CODES/GIT/DFT-Tools/SCF') 

import RKS

#### Choice of a density functional

Any KS calculation requires a choice of the approximation to the exchange-correlation (XC) energy functional. 

XC approximations can be used together or split into exchange-only (X) and correlation-only (C) functionals.

The list of all available approximate functionals is given in the LibXC website (https://libxc.gitlab.io/functionals/).

All functionals are given a LibXC name and an alias. The latter is how the functional is usually called in the literature or how you are used to be calling it. For example, "EXX" stands for the exact exchange functional, and "LDAx" stands for the local density approximation to the exchange energy. The first one is not given in LibXC (custom functional), the latter is listed as `LDA_X` in LibXC. 

To specify the functional, use a dictionary as follows:

In [ ]:
EXX = {
    "name": "EXX", 
    "x_hf": {"alpha": 1.00}, # fraction of the exact exchange (1.0 for exact exchange)
    "x_functionals": {"LDA_X": {"alpha": 0.00}}, # type and fraction of an approximate exchange functional (0.0 for exact exchange)
    "c_functionals": {"GGA_K_VW": {"alpha": 0.00}} # type and fraction of an approximate correlation functional (0.0 for exact exchange)
    }
LDAx = {
    "name": "LDAx",
    "x_hf": {"alpha": 0.00}, 
    "x_functionals": {"LDA_X": {"alpha": 1.00}},
    "c_functionals": {"LDA_C_VWN_RPA": {"alpha": 0.00}}
    }

The `"x_hf"` entry can be skipped if desired.

However, both `"x_functionals"` and `"x_functionals"` must be specified at all times. Skipping any of those will trigger an error. 

If you wish not to use an exchange or a correlation functional simply use `"alpha": 0.00`. 

In [16]:
psi4.core.set_output_file('output.dat', False)
mol = psi4.geometry(f"""
units bohr
0 1
O
H 1 1.81
H 1 1.81 2 104.5                    
symmetry c1
""")

method = LDAx

psi4.set_options({"basis": "def2-svp",
                  "DFT_SPHERICAL_POINTS": 302,
                  "DFT_RADIAL_POINTS": 350
                  })

E, D, homo, scf_iter = RKS.ks_solver(mol, method, damp=0.0, DIIS=True)


SCF converged in 12 iterations and 3.025 seconds


In [17]:
print(f"\nFinal SCF Energy: {E:.6f} Hartree")


Final SCF Energy: -75.130560 Hartree


In [18]:
psi4.set_options({"scf_type": "PK",
                  "d_convergence": 1e-8,
                  "e_convergence": 1e-8
                  })
psi4_E, psi4_wfn = psi4.energy('SCF', dft_functional=method, return_wfn=True)

In [19]:
print(f"Psi4 SCF Energy: {psi4_E:.6f} Hartree")

Psi4 SCF Energy: -75.130560 Hartree
